# 01 · Auditoria dos documentos de precedentes

Primeiro notebook da base de precedentes do TST. O `00_introducao_precedentes.ipynb` explica por que essa base existe. Aqui conferimos **o que foi recebido** antes de extrair qualquer texto.

**O que este notebook faz:**
1. **Inventário:** registra arquivo, emissor, data de referência, páginas, camada de texto e hash SHA-256 de cada PDF. O hash identifica a versão usada.
2. **Papel de cada documento:** define o que entra na base, o que serve só para conferir a vigência e o que vai para outra base.
3. **Extração bruta:** separa um bloco por súmula, OJ ou tema, sem limpeza.
4. **Integridade da extração:** confere se a numeração está completa e sem repetições.
5. **Vigência:** compara a coletânea, atualizada até 2024, com a notícia de cancelamentos.

**Ordem da pasta:**

| Notebook | Conteúdo | Situação |
|---|---|---|
| `00_introducao_precedentes` | Por que criar a terceira base: teste de posição e exemplos das 25 respostas | pronto |
| `01_auditoria_precedentes` | Este notebook | pronto |
| `02_eda_precedentes` | Volume, tamanho, situação, estrutura e normas citadas | pronto |
| `03_limpeza_estruturacao_precedentes` | Campos, vigência e o JSONL único | pronto |
| `04_chunking_precedentes` | Um trecho por enunciado ou item, com cabeçalho (1.268 trechos) | pronto |
| `05_embeddings_indexacao_precedentes` | BGE-M3 e schema `lex_precedentes` validado | pronto |
| `06_avaliacao_recuperacao_precedentes` | Hit@k com o precedente esperado | a fazer |

### Dados desta etapa

| | Caminho no repositório | Formato | Quantidade | Versionado? |
|---|---|---|---|---|
| Entrada | `data/raw/precedentes/*.pdf` | PDF | 10 arquivos (8 documentos distintos); coletânea TRT-3 com 421 páginas e índice de IRR com 221 | fora do Git |
| Saída | `data/interim/precedentes/verbetes_brutos.jsonl` | JSONL (um registro JSON por linha) | 1.596 blocos brutos: 1.172 súmulas e OJs + 424 blocos de temas (321 temas distintos) | fora do Git |
| Relatórios | `reports/precedentes/auditoria/` | CSV e JSON | inventario_documentos.csv (10 linhas), numeracao_series.csv (9), conferencia_noticia_cancelamento.csv (26), manifesto_auditoria.json | sim |

**Formato:** a base estruturada é gravada em **JSONL** (texto, um registro JSON por linha), e não em CSV, porque cada registro tem campos com listas (itens da súmula, páginas, assuntos, histórico). Os arquivos **CSV** do projeto são os relatórios em `reports/precedentes/`. Os arquivos de dados ficam em `data/`, dentro da pasta do repositório, mas **fora do Git** (`.gitignore`): contêm texto integral e são recriados pelos notebooks.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/precedentes').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)
RAW = ROOT / 'data/raw/precedentes'
BRUTOS = ROOT / 'data/interim/precedentes/verbetes_brutos.jsonl'
SAIDA_JSONL = ROOT / 'data/processed/precedentes/precedentes_tst.jsonl'
REL = ROOT / 'reports/precedentes'

from src.precedentes import documentos, extracao
(REL / 'auditoria').mkdir(parents=True, exist_ok=True)

## 1. Inventário dos arquivos

Os 10 PDFs enviados estão em `data/raw/precedentes/` exatamente como foram baixados. Eles ficam fora do Git porque são grandes: o maior tem 238 MB. O inventário registra o hash de cada um para que a extração possa ser refeita com a mesma versão.

Dois arquivos terminam em "(1)": a coletânea do TRT-3 e o índice de repercussão geral foram baixados duas vezes. O hash mostra que são cópias byte a byte dos originais. Eles ficam registrados como `duplicata` e não são lidos. Por isso, há **10 arquivos e 8 documentos distintos**.

> **Lê:** `data/raw/precedentes/*.pdf` (10 PDFs)  
> **Grava:** `reports/precedentes/auditoria/inventario_documentos.csv` (CSV, 10 linhas: uma por arquivo)

In [2]:
inv = documentos.inventario(RAW)
print(f'{len(inv)} PDFs na pasta · faltando do catálogo: {inv.attrs["faltando"] or "nenhum"}')
repetidos = inv[inv.sha256.duplicated(keep=False)].sort_values('sha256')
print(f'Arquivos com hash repetido: {len(repetidos)} · documentos distintos: {inv.sha256.nunique()}')
print('Duplicatas confirmadas pelo hash:', ', '.join(repetidos[repetidos.papel == "duplicata"].sigla))
display(inv[['sigla', 'emissor', 'data_referencia', 'paginas', 'tamanho_mb', 'camada_texto', 'paginas_sem_texto', 'papel']]
        .assign(sha256=inv.sha256.str[:12] + '…'))
inv.drop(columns=['uso']).to_csv(REL / 'auditoria/inventario_documentos.csv', index=False, encoding='utf-8')

10 PDFs na pasta · faltando do catálogo: nenhum
Arquivos com hash repetido: 4 · documentos distintos: 8
Duplicatas confirmadas pelo hash: TRT3_2024_COPIA, RG_STF_2026_COPIA


,sigla,emissor,data_referencia,paginas,tamanho_mb,camada_texto,paginas_sem_texto,papel,sha256
0,ACORDAO_168,TST,NaN,8,0.1,True,0,referencia,2eba0baa6d93…
1,TRT3_2024_COPIA,TRT da 3ª Região (SEJPAC),2024-06-28,421,6.4,True,4,duplicata,603999652e74…
2,TRT3_2024,TRT da 3ª Região (SEJPAC),2024-06-28,421,6.4,True,4,fonte_principal,603999652e74…
3,NOTICIA_CANCELAMENTO,não oficial (notícia),NaN,5,0.1,True,0,fonte_vigencia,eb8a1f39822d…
4,AUTOGRAFO_EC_2026,Câmara dos Deputados / Senado Federal,2026-05-28,6,0.4,True,0,fora_do_escopo,8d0b9303d395…
5,TRT2_2021,TRT da 2ª Região,2021-12-06,441,227.1,True,1,fora_do_escopo,ddc1e9939285…
6,EC_141_2026,Presidência da República (Planalto),2026-10-16,3,1.0,False,3,fora_do_escopo,1be36f847ff9…
7,RG_STF_2026_COPIA,TST (Gabinete da Presidência),2026-11-19,377,4.1,True,0,duplicata,94e4615494d2…
8,RG_STF_2026,TST (Gabinete da Presidência),2026-11-19,377,4.1,True,0,referencia,94e4615494d2…
9,IRR_2026,TST,2026-11-19,221,1.8,True,1,fonte_principal,3fd7edd1c4d4…


## 2. O papel de cada documento

Os documentos têm três destinos diferentes. A decisão fica registrada aqui e pode ser revista.

In [3]:
display(inv[['sigla', 'papel', 'destino', 'uso']])

,sigla,papel,destino,uso
0,ACORDAO_168,referencia,nenhuma (dados pessoais),Exemplo de reafirmação de tese em IRR. Contém nomes das partes; a tese já está no índice de IRR.
1,TRT3_2024_COPIA,duplicata,nenhuma (cópia de TRT3_2024),Mesmo hash de arquivo-versao-eletronica-28-6-2024-1.pdf; não é lida na extração.
2,TRT3_2024,fonte_principal,base de precedentes,"Súmulas e OJs do TST (2ª parte, p. 65-256), no formato oficial com situação e histórico de cada verbete."
3,NOTICIA_CANCELAMENTO,fonte_vigencia,base de precedentes (só alerta),"Lista de enunciados noticiados como cancelados depois de 06/2024; vira alerta de vigência, não altera a situação ofi..."
4,AUTOGRAFO_EC_2026,fora_do_escopo,nenhuma (versão de tramitação),Texto em tramitação da mesma emenda; a versão oficial é a EC 141.
5,TRT2_2021,fora_do_escopo,nenhuma,Mais antiga que a coletânea do TRT-3 e redundante para o TST.
6,EC_141_2026,fora_do_escopo,base de legislação (atualizar o art. 7º da CF),"Não é precedente: é norma constitucional. Página em imagem, sem camada de texto."
7,RG_STF_2026_COPIA,duplicata,nenhuma (cópia de RG_STF_2026),Mesmo hash do índice de repercussão geral; não é lida.
8,RG_STF_2026,referencia,etapa posterior (precedentes do STF),"Repercussão geral, súmulas vinculantes e controle concentrado do STF. Fica para a etapa do STF."
9,IRR_2026,fonte_principal,base de precedentes,"Temas repetitivos (parte temática, p. 12-121): questão jurídica, tese e referências; situação pela cor do título."


**Resumo das decisões:**

- **Entram na base:**
  - **Coletânea do TRT-3 (06/2024):** súmulas e OJs do TST. É a única fonte recebida que traz todos os verbetes no formato oficial, com situação e histórico.
  - **Índice de IRR do TST (19/11/2026):** temas repetitivos.
- **Serve só como alerta de vigência:** a **notícia de cancelamento**. Não é documento oficial e foi gerada em 22/11/2026 por uma ferramenta automática, sem data de publicação nem link para a resolução. Por isso não altera a situação oficial (seção 5).
- **Ficam para depois:**
  - **Índice de repercussão geral do STF:** entra na etapa do STF.
  - **Acórdão 168:** a tese dele, a reafirmação da Súmula 462, já está no índice de IRR, e o documento contém nomes das partes.
- **Não entram:**
  - **Coletânea do TRT-2 (2021):** é mais antiga e repete a do TRT-3.
  - **EC 141/2026 e autógrafo da Câmara:** são norma constitucional, não precedente. A EC 141 foi aplicada à **base de legislação** numa versão nova (`ec141_2026`), com o art. 7º atualizado e os 10 artigos da emenda (`scripts/atualizar_legislacao_ec141.py`). A cópia bruta da Constituição, mostrada na célula abaixo, não foi editada e guarda a redação anterior.
- **Cópias "(1)":** duas cópias idênticas, registradas como `duplicata`.

In [4]:
cf = ROOT / 'data/raw/legislacao/constituicao/constituicao-1988.md'
if cf.exists():
    linhas = [l.strip() for l in cf.read_text(encoding='utf-8').splitlines() if l.startswith(('XIII - duração', 'XV - repouso'))]
    print('Art. 7º na cópia bruta da Constituição (redação anterior; a busca usa a versão ec141_2026):')
    for l in linhas:
        print('  ', l[:150])
else:
    print('Cópia local da legislação não encontrada; conferir na base de legislação.')

Art. 7º na cópia bruta da Constituição (redação anterior; a busca usa a versão ec141_2026):
   XIII - duração do trabalho normal não superior a oito horas diárias e quarenta e quatro semanais, facultada a compensação de horários e a redução da j
   XV - repouso semanal remunerado, preferencialmente aos domingos;


## 3. Que parte de cada PDF é usada

| Documento | Páginas | Conteúdo | Fica de fora |
|---|---|---|---|
| Coletânea TRT-3 | 65-256 | Súmulas (65-152), OJs do Tribunal Pleno/Órgão Especial (154-155), SBDI-1 (157-212), SBDI-1 transitória (214-225), SBDI-2 (227-250) e SDC (252-256) | A 1ª parte (TRT-3) e os temas de IRR de 2024, substituídos pelo índice de 2026. Também os precedentes normativos e os índices |
| Índice de IRR | 12-121 | Parte temática: cada tema com título, questão jurídica, tese e referências | Apresentação, glossário e a tabela final, que repete os mesmos temas em colunas |

Cada verbete da coletânea começa com o código oficial (`SUM-338`, `OJ-SDI1-394`...). No índice de IRR, a **cor do título** indica a situação do tema, segundo a legenda da página 2:
- azul: tese fixada;
- mostarda: afetado, ainda sem tese;
- vermelho: tese superada.

In [5]:
pag = extracao.texto_paginas(RAW / 'arquivo-versao-eletronica-28-6-2024-1.pdf', [4])[4]
print(pag[pag.find('2ª PARTE'):pag.find('ÍNDICES')].strip())

2ª PARTE  
TRIBUNAL SUPERIOR DO TRABALHO  
Remissão à jurisprudência consolidada do TRT da 3ª Região 
SÚMULAS...........................................................................................................................65 - 152 
ORIENTAÇÕES JURISPRUDENCIAIS DO TRIBUNAL PLENO/ÓRGÃOESPECIAL......154 - 155 
ORIENTAÇÕES JURISPRUDENCIAIS DA SBDI-1..........................................................157 - 212 
ORIENTAÇÕES JURISPRUDENCIAIS DA SBDI-1-TRANSITÓRIA.................................214 - 225 
ORIENTAÇÕES JURISPRUDENCIAIS DA SBDI-2..........................................................227 - 250 
ORIENTAÇÕES JURISPRUDENCIAIS DA SDC..............................................................252 - 256 
TEMAS DE IRR.................................................................................................................258 - 266 
PRECEDENTES NORMATIVOS.......................................................................................268 - 275


## 4. Extração bruta e integridade da numeração

A extração só separa os blocos, sem limpar nada. A limpeza é feita no notebook 03. A numeração de cada série deve ser contínua, de 1 até o maior número, sem faltas nem repetições. Os cancelados continuam listados na coletânea com o próprio número, então uma falta indicaria um verbete perdido na leitura.

Durante o desenvolvimento, essa conferência pegou três problemas na leitura do PDF, já corrigidos:
- códigos colados no título, como `SUM-418MANDADO` e `SUM- 43`;
- redações antigas no histórico que pareciam verbetes novos, como `OJ-SDC-6 –Dissídio...`;
- títulos de tema justificados palavra por palavra, em que o número ficava em outro bloco.

> **Lê:** a coletânea do TRT-3 (p. 65–256) e o índice de IRR (p. 12–121)  
> **Grava:** `reports/precedentes/auditoria/numeracao_series.csv` (CSV, 9 linhas: uma por série)

In [6]:
verbetes = extracao.verbetes_tst_trt3(RAW / 'arquivo-versao-eletronica-28-6-2024-1.pdf')
temas = extracao.temas_irr(RAW / 'Índice Temático IRRs - versão Novembro 19_11_2026.pdf')

def numeracao(registros, chave):
    linhas = []
    for serie, grupo in pd.DataFrame(registros).groupby(chave):
        nums = grupo.numero.tolist()
        faltam = sorted(set(range(1, max(nums) + 1)) - set(nums))
        linhas.append(dict(serie=serie, blocos=len(nums), distintos=len(set(nums)), maior_numero=max(nums),
                           faltando=len(faltam), numeros_faltando=', '.join(map(str, faltam[:20])) or '-'))
    return pd.DataFrame(linhas)

num_v = numeracao(verbetes, 'serie')
num_t = numeracao(temas, 'subtipo')
print(f'Súmulas e OJs: {len(verbetes)} blocos · temas: {len(temas)} blocos ({len({t["codigo"] for t in temas})} temas distintos)')
display(num_v)
display(num_t)
pd.concat([num_v.assign(fonte='TRT3_2024'), num_t.assign(fonte='IRR_2026')]).to_csv(
    REL / 'auditoria/numeracao_series.csv', index=False, encoding='utf-8')

Súmulas e OJs: 1172 blocos · temas: 424 blocos (321 temas distintos)


,serie,blocos,distintos,maior_numero,faltando,numeros_faltando
0,OJ-SDC,38,38,38,0,-
1,OJ-SDI1,421,421,421,0,-
2,OJ-SDI1T,79,79,79,0,-
3,OJ-SDI2,158,158,158,0,-
4,OJ-TP/OE,13,13,13,0,-
5,SUM,463,463,463,0,-


,serie,blocos,distintos,maior_numero,faltando,numeros_faltando
0,IAC,2,2,3,1,1
1,IRDR,4,2,2,0,-
2,IRR,418,317,317,0,-


**Leitura:**
- **Súmulas e OJs:** as seis séries estão completas, sem faltas nem repetições. Cada número de 1 ao último aparece uma única vez.
- **Temas repetitivos:** os 317 IRR e os 2 IRDR estão completos. No IAC, a contagem mostra o nº 1 como faltante, mas o próprio índice remissivo do PDF só lista os IAC 002 e 003. O nº 1 não está no documento, então não houve perda na leitura. Há mais blocos do que temas porque o índice repete o tema em cada assunto a que ele pertence e na seção de suspensões. O notebook 03 junta essas repetições.

## 5. Vigência: a coletânea de 2024 diante da notícia de cancelamentos

A coletânea foi atualizada até **28/06/2024**. A notícia recebida diz que o TST cancelou 36 enunciados, mas só nomeia 26. Conferimos cada um desses 26 na coletânea:
- se o enunciado existe;
- se a coletânea já o dava como cancelado.

> **Lê:** `data/raw/precedentes/cancelamento_36_enunciados_tst.pdf`  
> **Grava:** `reports/precedentes/auditoria/conferencia_noticia_cancelamento.csv` (CSV, 26 linhas)

In [7]:
noticia = pd.DataFrame(extracao.cancelamentos_noticia(RAW / 'cancelamento_36_enunciados_tst.pdf'))
cabecalho = {v['codigo']: v['texto_bruto'][:400].lower() for v in verbetes}
noticia['na_coletanea'] = noticia.codigo.map(lambda c: c in cabecalho)
noticia['ja_cancelado_em_2024'] = noticia.codigo.map(lambda c: 'cancelad' in ' '.join(cabecalho.get(c, '').split('\n')[:3]))
print(f'Itens nomeados na notícia: {len(noticia)} · na coletânea: {noticia.na_coletanea.sum()} '
      f'(o PN 100 é precedente normativo, fora do escopo) · já cancelados em 2024: {noticia.ja_cancelado_em_2024.sum()}')
display(noticia[['codigo', 'titulo_noticia', 'alcance', 'na_coletanea', 'ja_cancelado_em_2024']])
noticia.to_csv(REL / 'auditoria/conferencia_noticia_cancelamento.csv', index=False, encoding='utf-8')

Itens nomeados na notícia: 26 · na coletânea: 25 (o PN 100 é precedente normativo, fora do escopo) · já cancelados em 2024: 0


,codigo,titulo_noticia,alcance,na_coletanea,ja_cancelado_em_2024
0,PN-100,Férias iniciadas em sábado,integral,False,False
1,SUM-450,Férias pagas com atraso,integral,True,False
2,OJ-SDI1-14,Aviso prévio cumprido em casa,integral,True,False
3,OJ-SDI1-355,Intervalo interjornada,integral,True,False
4,SUM-452,Promoção por plano de cargos (PCS),integral,True,False
5,OJ-SDI1-383,Isonomia entre contratados diretos e terceirizados,integral,True,False
6,SUM-449,Minutos que antecedem ou sucedem a jornada:,integral,True,False
7,SUM-444,Jornada 12×36,integral,True,False
8,SUM-437,Intervalo intrajornada,integral,True,False
9,SUM-6,Equiparação Salarial,"itens I, II, VI (alínea “b”) e X.",True,False


**Leitura:** quase todos os enunciados da notícia aparecem como **vigentes** na coletânea de 2024. Ou seja, se a notícia estiver certa, a coletânea está desatualizada justamente em temas comuns nas perguntas:
- intervalo intrajornada (Súmula 437);
- jornada 12x36 (Súmula 444);
- horas in itinere (Súmula 90);
- terceirização (Súmula 331, I);
- preposto (Súmula 377).

**Decisão (aplicada no notebook 03):**
- a situação oficial continua sendo a da coletânea;
- cada enunciado citado na notícia recebe um **alerta de vigência**;
- os enunciados dados como cancelados **por inteiro** ficam **fora da busca** até a resolução oficial do TST ser conferida. Citar como regra uma súmula cancelada é pior do que não ter a súmula;
- os enunciados com cancelamento **parcial**, como a Súmula 6 em alguns itens e as Súmulas 331 e 372 no item I, continuam na busca, com o alerta.

**Pendente:** o livro oficial de súmulas e OJs do TST atualizado em 2026, ou a resolução dos cancelamentos. Com ele, este notebook roda de novo e os alertas viram situação oficial.

## 6. Gravação dos blocos brutos e manifesto

Os blocos brutos vão para `data/interim/precedentes/`, fora do Git, e são a entrada da EDA e da limpeza. O manifesto, que é versionado, registra o hash de cada PDF de origem e o hash do arquivo bruto. Com isso, qualquer mudança nos documentos ou no extrator aparece na próxima execução.

> **Grava:** `data/interim/precedentes/verbetes_brutos.jsonl` (JSONL, 1.596 linhas, fora do Git) e `reports/precedentes/auditoria/manifesto_auditoria.json` (hashes)

In [8]:
import hashlib
BRUTOS.parent.mkdir(parents=True, exist_ok=True)
sha = dict(zip(inv.arquivo, inv.sha256))
with BRUTOS.open('w', encoding='utf-8') as f:
    for v in verbetes:
        f.write(json.dumps(dict(v, fonte='TRT3_2024'), ensure_ascii=False) + '\n')
    for t in temas:
        f.write(json.dumps(dict(t, fonte='IRR_2026'), ensure_ascii=False) + '\n')
manifesto = dict(
    fontes={s: dict(arquivo=a, sha256=sha[a]) for s, a in [('TRT3_2024', 'arquivo-versao-eletronica-28-6-2024-1.pdf'),
                                                          ('IRR_2026', 'Índice Temático IRRs - versão Novembro 19_11_2026.pdf'),
                                                          ('NOTICIA_CANCELAMENTO', 'cancelamento_36_enunciados_tst.pdf')]},
    paginas=dict(TRT3_2024=extracao.PAGINAS_TST_TRT3, IRR_2026=extracao.PAGINAS_TEMAS_IRR),
    blocos=dict(verbetes=len(verbetes), temas=len(temas)),
    brutos_sha256=hashlib.sha256(BRUTOS.read_bytes()).hexdigest())
(REL / 'auditoria/manifesto_auditoria.json').write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

{
  "fontes": {
    "TRT3_2024": {
      "arquivo": "arquivo-versao-eletronica-28-6-2024-1.pdf",
      "sha256": "603999652e74bac701b518dce3bb87b31cf12d086ffc7074c8b57ed6e46afbbc"
    },
    "IRR_2026": {
      "arquivo": "Índice Temático IRRs - versão Novembro 19_11_2026.pdf",
      "sha256": "3fd7edd1c4d41143815b6df4cdd891eac688e5dcb104936102c519cbeb1f9985"
    },
    "NOTICIA_CANCELAMENTO": {
      "arquivo": "cancelamento_36_enunciados_tst.pdf",
      "sha256": "eb8a1f39822d03090505c85b759ce76f0f765ec225a2c69ceabbb023b5bed74e"
    }
  },
  "paginas": {
    "TRT3_2024": [
      65,
      256
    ],
    "IRR_2026": [
      12,
      121
    ]
  },
  "blocos": {
    "verbetes": 1172,
    "temas": 424
  },
  "brutos_sha256": "bcee3c2b1bb63f6b7faa8ac908f34c0e00b6bc6e1b53162cc360f409b4c0c2a4"
}


## Conclusões

- **Recebido e conferido:** 10 PDFs, com hash. São 8 documentos distintos e 2 cópias idênticas, assim distribuídos:
  - 2 fontes principais;
  - 1 fonte de vigência;
  - 2 referências;
  - 3 fora do escopo.
- **Extração íntegra:**
  - 463 súmulas e 709 OJs, em seis séries completas;
  - 321 temas repetitivos (317 IRR, 2 IAC e 2 IRDR), sem falta de numeração. O IAC 1 não consta do índice.
- **Risco principal:** a vigência. A coletânea é de 06/2024, e a notícia aponta cancelamentos posteriores. A solução provisória é a quarentena; a definitiva é o livro oficial de 2026.
- **Fora desta base:** a EC 141/2026, que foi aplicada à **base de legislação**. O art. 7º da Constituição ganhou uma versão nova, `ec141_2026`, com a jornada de 40 horas progressiva e os dois dias de repouso. A célula da seção 2 mostra a redação antiga, que continua guardada na versão `original`.

Próximo: `02_eda_precedentes.ipynb`.